# Baseline Models for Quora Question Pairs

This notebook demonstrates baseline approaches to duplicate question detection.

## Setup and Data Loading

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Add src to path
sys.path.insert(0, '..')

from src import (
    load_data,
    engineer_features,
    LogisticRegressionModel,
    evaluate_model,
    print_evaluation_report
)

%matplotlib inline
sns.set_style('whitegrid')

### Load Data

In [ ]:
# Load datasets
df_train, df_test = load_data(
    '../data/quora_question_pairs_train.csv.zip',
    '../data/quora_question_pairs_test.csv.zip'
)

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")
print(f"\nDuplicate distribution:\n{df_train['is_duplicate'].value_counts()}")

### Engineer Features

In [ ]:
# Engineer features for both sets
df_train = engineer_features(df_train)

# Check engineered features
feature_cols = ['q1_len', 'q2_len', 'len_diff', 'common_words', 'jaccard_sim', 'word_match_share']
print("Feature statistics:")
print(df_train[feature_cols].describe())

## Baseline: Logistic Regression

### Prepare Train/Test Split

In [ ]:
# Prepare features and target
X = df_train[feature_cols]
y = df_train['is_duplicate']

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Train duplicate ratio: {y_train.mean():.2%}")
print(f"Validation duplicate ratio: {y_val.mean():.2%}")

### Train Logistic Regression Model

In [ ]:
# Train model
lr_model = LogisticRegressionModel(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

# Make predictions
y_pred = lr_model.predict(X_val)
y_pred_proba = lr_model.predict_proba(X_val)

# Evaluate
metrics = evaluate_model(y_val, y_pred, y_pred_proba)
print_evaluation_report(metrics)

### Feature Importance

In [ ]:
# Get feature coefficients
coefficients = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr_model.model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print(coefficients)

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(coefficients['feature'], coefficients['coefficient'])
plt.xlabel('Coefficient')
plt.title('Feature Importance (Logistic Regression)')
plt.tight_layout()

## Summary

The baseline Logistic Regression model provides a simple yet effective approach to duplicate detection.
Key observations:
- Word matching and similarity features are strong predictors
- The model captures the relationship between text overlap and duplicate probability